In [5]:
import numpy as np
import pandas as pd
import sklearn
import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import f1_score, precision_score, accuracy_score, recall_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [3]:
X_train, y_train = train_df["Combined_Review"], train_df["source"]
X_test,  y_test  = test_df["Combined_Review"],  test_df["source"]
cv = StratifiedKFold(n_splits= 5, shuffle= True, random_state= 42)

In [4]:
vectorizer = CountVectorizer()

**Single Classification Tree**

In [7]:
ct_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(
        min_df=2,
        max_df=0.95
    )),
    ('top_k', SelectKBest(chi2, k=1000)),
    ('ct', DecisionTreeClassifier(
        random_state=42
    ))
])

ct_pipeline.fit(X_train, y_train)

Pipeline(steps=[('vectorizer', CountVectorizer(max_df=0.95, min_df=2)),
                ('top_k',
                 SelectKBest(k=1000,
                             score_func=<function chi2 at 0x000001EFE7055580>)),
                ('ct', DecisionTreeClassifier(random_state=42))])

In [12]:
param_grid_ct = {
    "ct__criterion": ['gini', 'entropy', 'log_loss'],
    'ct__splitter': ['best', 'random'],
    'ct__max_depth': [5, 10, 15, 20],
    'ct__min_samples_split': [2,5,10,15],
    'ct__min_samples_leaf': [1,2,4,8],
    'ct__max_features': [None, 'sqrt', 'log2']
}

grid_ct = GridSearchCV(
    ct_pipeline,
    param_grid_ct,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid_ct.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('vectorizer',
                                        CountVectorizer(max_df=0.95, min_df=2)),
                                       ('top_k',
                                        SelectKBest(k=1000,
                                                    score_func=<function chi2 at 0x000001EFE7055580>)),
                                       ('ct',
                                        DecisionTreeClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'ct__criterion': ['gini', 'entropy', 'log_loss'],
                         'ct__max_depth': [5, 10, 15, 20],
                         'ct__max_features': [None, 'sqrt', 'log2'],
                         'ct__min_samples_leaf': [1, 2, 4, 8],
                         'ct__min_samples_split': [2, 5, 10, 15],
                         'ct__splitter': ['best', 'random']},
             scoring='accuracy')

In [15]:
print(grid_ct.best_params_)
best_ct = grid_ct.best_estimator_

{'ct__criterion': 'entropy', 'ct__max_depth': 20, 'ct__max_features': None, 'ct__min_samples_leaf': 1, 'ct__min_samples_split': 15, 'ct__splitter': 'random'}


In [16]:
y_pred_ct = best_ct.predict(X_test)
f1_ct = f1_score(y_test, y_pred_ct)
acc_ct = accuracy_score(y_test, y_pred_ct)
recall_ct = recall_score(y_test, y_pred_ct)
pre_ct = precision_score(y_test, y_pred_ct)
print('rf result')
print('f1',f1_ct)
print('-----')
print('accuracy: ', acc_ct)
print('-----')
print('recall:', recall_ct)
print('-----')
print('precision: ', pre_ct)
print('-----')

rf result
f1 0.8132780082987552
-----
accuracy:  0.82
-----
recall: 0.784
-----
precision:  0.8448275862068966
-----


In [17]:
joblib.dump(best_ct, 'models/ct.joblib')


['models/ct.joblib']